In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!unzip -q "/content/drive/MyDrive/KhPI_HW/linear_algebra_intro_practice-main.zip" -d /content/idpp
!find /content/idpp -name "matrix_decomposition.py"
!find /content/idpp -name "pyproject.toml"

/content/idpp/linear_algebra_intro_practice-main/matrix_decomposition.py
/content/idpp/linear_algebra_intro_practice-main/pyproject.toml


In [3]:
%cd /content/idpp/linear_algebra_intro_practice-main

/content/idpp/linear_algebra_intro_practice-main


In [4]:
!pip -q install uv
!uv sync
!uv run pytest tests/test_matrix_decomposition.py -v

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 37.0 MB/s eta 0:00:00
Using CPython 3.13.16 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 113 packages in 863ms
Prepared 109 packages in 5.84s
Installed 109 packages in 429ms
 + anyio==4.15.1
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==26.1.0
 + arrow==1.4.0
 + asttokens==3.0.2
 + async-lru==2.4.0
 + attrs==26.1.0
 + babel==2.18.0
 + beautifulsoup4==4.15.0
 + bleach==6.4.0
 + certifi==2026.10.19
 + cffi==2.1.1
 + charset-normalizer==3.5.2
 + cloudpickle==3.1.2
 + comm==0.2.3
 + contourpy==1.4.0
 + cycler==0.12.1
 + debugpy==1.8.22
 + defusedxml==0.7.1
 + executing==2.3.0
 + fastjsonschema==2.22.2
 + fonttools==4.67.0
 + fqdn==1.6.0
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.20
 + iniconfig==2.3.1
 + ipykernel==7.3.0
 + ipython==9.17.1
 + ipython-pygments-lexers==1.1.1
 + isoduration==20.11.0
 + jedi==0.20.1
 + jinja2==3.1.6
 + joblib==1.6.0
 + json5==0.17.3
 + jsonpointer=

In [9]:
# Перезавантажуємо модуль, щоб Colab точно взяв останню версію matrix_decomposition.py
import importlib
import numpy as np
import matrix_decomposition

importlib.reload(matrix_decomposition)


def test_lu_decomposition_property():
    # Перевіряємо головну властивість LU: P@L@U має дорівнювати вихідній матриці x
    x = np.array([[2.0, 1.0], [1.0, 3.0]])
    P, L, U = matrix_decomposition.lu_decomposition(x)

    reconstructed = P @ L @ U
    np.testing.assert_allclose(reconstructed, x, atol=1e-7)
    assert L.shape == x.shape, "Розмірність L повинна збігатися з розмірністю x"
    assert U.shape == x.shape, "Розмірність U повинна збігатися з розмірністю x"

def test_qr_decomposition_rectangular():
    # Дослідж.: неквадратна матрица (3x2). Перевіряємо mode="full" та ортогональність Q
    x = np.array([[1.0, -1.0], [2.0, 0.0], [0.0, 1.0]])
    m, n = x.shape

    Q, R = matrix_decomposition.qr_decomposition(x)

    # Q має бути розміру (m, m), R — (m, n)
    assert Q.shape == (m, m), f"Очікувалася розміру Q (3, 3), отримано {Q.shape}"
    assert R.shape == (m, n), f"Очікувалася розміру R (3, 2), отримано {R.shape}"

    # Перевірка реконструкції Q @R == x
    np.testing.assert_allclose(Q @ R, x, atol=1e-7)

    # Перевірка ортогональності Q (Q.T@ Q має давати одиничну матрицю)
    np.testing.assert_allclose(Q.T @ Q, np.eye(m), atol=1e-7)

def test_determinant_edge_cases():
    # Дослідж.: одинична матриця (det = 1) та лінійно залежна/вироджена (det = 0)
    identity = np.eye(3)
    np.testing.assert_allclose(matrix_decomposition.determinant(identity), 1.0, atol=1e-7)

    # Рядки лінійно залежні -> визначник дорів. 0
    singular = np.array([[1.0, 2.0], [2.0, 4.0]])
    np.testing.assert_allclose(matrix_decomposition.determinant(singular), 0.0, atol=1e-7)

def test_determinant_simple_2x2():
    # стандартн. тест для визначника простої матриці 2x2
    x = np.array([[2.0, 3.0], [1.0, 4.0]])
    # Визначник: (2 * 4) - (3 * 1) = 8 - 3 = 5.0
    result = matrix_decomposition.determinant(x)
    np.testing.assert_allclose(result, 5.0, atol=1e-7)

def test_eigen_decomposition_property():
    # Перевіряємо зв'язок власних значень і векторів: A@v = lambda*v
    x = np.array([[4.0, -2.0], [1.0, 1.0]])
    vals, vecs = matrix_decomposition.eigen(x)

    for i in range(x.shape[0]):
        v = vecs[:, i:i+1]  # Власний вектор (стовпець)
        lam = vals[i]  # Власне значення
        np.testing.assert_allclose(x @ v, lam * v, atol=1e-7)

def test_svd_reconstruction():
    # Дослідж.: правильн. реконструкція SVD для прямокутної матриці (враховуючи що V = Vt.T)
    x = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
    U, S, V = matrix_decomposition.svd(x)

    # Формуємо діаг. матрицю Sigma правильного розміру (тобто такого ж як x)
    Sigma = np.zeros(x.shape, dtype=float)
    np.fill_diagonal(Sigma, S)

    # ми поверт. V = Vt.T, то транспонірувавши V назад (V.T),
    # ми отримуємо оригінальний Vt для множення: U @ Sigma @ V.T
    reconstructed = U @ Sigma @ V.T
    np.testing.assert_allclose(reconstructed, x, atol=1e-7)

def test_svd_singular_values_non_negative():
    # Перевіряємо властивість SVD: синглярні значення (S) завжди мають бути >= 0
    x = np.array([[1.0, 2.0], [3.0, 4.0]])
    _, S, _ = matrix_decomposition.svd(x)
    assert np.all(S >= 0), "Синглярні значення за визначенням не можуть бути від'ємними"

# Список усіх тестів для запуску
tests = [
    test_lu_decomposition_property,
    test_qr_decomposition_rectangular,
    test_determinant_edge_cases,
    test_determinant_simple_2x2,
    test_eigen_decomposition_property,
    test_svd_reconstruction,
    test_svd_singular_values_non_negative,
]

failed = []

for test in tests:
    try:
        test()
        print(f"PASS: {test.__name__}")
    except Exception as error:
        failed.append((test.__name__, str(error)))
        print(f"FAIL: {test.__name__}\n  {error}")

if failed:
    raise AssertionError(f"\nНе пройдено тестів: {len(failed)}")

print(f"\nУсі {len(tests)} локальних тестів для розкладу матриць пройдено.")

PASS: test_lu_decomposition_property
PASS: test_qr_decomposition_rectangular
PASS: test_determinant_edge_cases
PASS: test_determinant_simple_2x2
PASS: test_eigen_decomposition_property
PASS: test_svd_reconstruction
PASS: test_svd_singular_values_non_negative

Усі 7 локальних тестів для розкладу матриць пройдено.


# Код для matrix_decomposition

In [ ]:
import numpy as np
import scipy.linalg

def lu_decomposition(x: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Perform LU decomposition of a matrix.

    Args:
        x (np.ndarray): The input matrix to decompose.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]:
            The permutation matrix P, lower triangular matrix L, and upper triangular matrix U.
    """
    #raise NotImplementedError
    # використ. scipy.linalg для LU-розкладу
    P, L, U = scipy.linalg.lu(x)
    return P, L, U

def qr_decomposition(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Perform QR decomposition of a matrix.

    Args:
        x (np.ndarray): The input matrix to decompose.

    Returns:
        tuple[np.ndarray, np.ndarray]: The orthogonal matrix Q and upper triangular matrix R.
    """
    # raise NotImplementedError
    # обчисл. повний QR-розклад (mode='full')
    # для запобігання помилок розмірності на неквадратних матрицях
    Q, R = scipy.linalg.qr(x, mode="full")
    return Q, R

def determinant(x: np.ndarray) -> np.ndarray:
    """
    Calculate the determinant of a matrix.

    Args:
        x (np.ndarray): The input matrix.

    Returns:
        np.ndarray: The determinant of the matrix.
    """
    #raise NotImplementedError
    # Використ. slogdet для обчислення визначника через логарифм,
    # що захищ. від переповнення inf або занулення в погано обумовлених вхід. матриць
    sign, logdet = np.linalg.slogdet(x)
    return sign * np.exp(logdet)

def eigen(x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Compute the eigenvalues and right eigenvectors of a matrix.

    Args:
        x (np.ndarray): The input matrix.

    Returns:
        tuple[np.ndarray, np.ndarray]: The eigenvalues and the right eigenvectors of the matrix.
    """
    #raise NotImplementedError
    #автоматич. балансує матрицю для мінімізації похибок округлення у погано обумов. матрицях
    eigenval, eigenvec = scipy.linalg.eig(x)
    return eigenval, eigenvec

def svd(x: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Perform Singular Value Decomposition (SVD) of a matrix.

    Args:
        x (np.ndarray): The input matrix to decompose.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: The matrices U, S, and V.
    """
    #raise NotImplementedError
    # Рахуємо повний розклад з full_matrices=True,
    #щоб отримати квадратні матриці й уникнути помилок з розмірністю
    U, S, Vt = np.linalg.svd(x, full_matrices=True)
    # +транспон. Vt у V
    return U, S, Vt.T